# Parameter sensitive analysis

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import pickle

from thesis.correlation_sociodemographic_covid.util import save_model, summarize_results, \
    tunning_negative_binomial_model

## Loading data

In [3]:
df_variables = pd.read_csv('data/df_without_collinearity_standardized.csv', index_col=0)
df_variables = df_variables.drop(columns=['percentage_population_age_range_60_more', 'percentage_male_population'])

In [4]:
df_political = pd.read_csv('data/df_political_without_missing_points.csv', index_col=0)[['percentual_votes_for_bolsonaro']]

In [5]:
df_y = pd.read_csv('data/df_mortality.csv', index_col=0)

In [6]:
df_cluster_probabilities = pd.read_csv('data/df_standardized_pca_2spherical_5_probability.csv', index_col=0)
df_cluster_probabilities.columns = ['Semi-urbanized', 'Urbanized', 'Rural with high human development', 'Urbanized with informal settlements', 'Rural with low human development']

## Parameter Sensitivity Analysis

In [8]:
list_columns_y = ['expected_deaths_1s_2020', 'expected_deaths_2020', 'expected_deaths_2021', 'expected_deaths_2022', 'expected_deaths_entire']
list_periods = ['2020_1','2020', '2021', '2022', '2020_2022']

for sample in range(30):
    print('\n*** Sample: ', sample)
    for i in range(len(list_periods)):
        column_y = list_columns_y[i]
        period = list_periods[i]
        print('\n*** Period: ', period)
            
        y = df_y[column_y]

        # print('\*** Model 6')
        # print('===>Full model:')
        # with open('models/model_6_'+period+'.pkl', 'rb') as file:
        #     model = pickle.load(file)
        # summarize_results(model)
        #
        # x = df_cluster_probabilities.drop(columns=['Urbanized']).copy()
        # x = sm.add_constant(x)
        #
        # x = x.loc[y>0]
        # y = y.loc[y>0]
        #
        # x_bootstrap = x.sample(frac=1, replace=True)
        # y_bootstrap = y.loc[x_bootstrap.index]
        # model = tunning_negative_binomial_model(x_bootstrap,y_bootstrap,list_offset=None)
        # filename = 'model_6_'+period+'_sample_'+str(sample)
        # save_model(model,filename,'models/sensitivity_analysis/parameter')
        # summarize_results(model)

        # print('\*** Model 11')
        print('===>Full model:')
        with open('models/model_11_'+period+'.pkl', 'rb') as file:
            model = pickle.load(file)
        summarize_results(model)

        x = df_variables.copy()
        scaler = StandardScaler()
        percentage_votes_for_bolsonaro_standardized = scaler.fit_transform(df_political)
        x['percentage_votes_for_bolsonaro'] = percentage_votes_for_bolsonaro_standardized[:,0]
        x = sm.add_constant(x)

        x = x.loc[y>0]
        y = y.loc[y>0]

        x_bootstrap = x.sample(frac=1, replace=True)
        y_bootstrap = y.loc[x_bootstrap.index]
        model = tunning_negative_binomial_model(x_bootstrap,y_bootstrap,list_offset=None)
        filename = 'model_11_'+period+'_sample_'+str(sample)
        save_model(model,filename,'models/sensitivity_analysis/parameter')
        summarize_results(model)


*** Sample:  0

*** Period:  2020_1
===>Full model:
                    Generalized Linear Model Regression Results                    
Dep. Variable:     expected_deaths_1s_2020   No. Observations:                 3045
Model:                                 GLM   Df Residuals:                     3030
Model Family:             NegativeBinomial   Df Model:                           14
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:                -36078.
Date:                     Fri, 20 Jun 2025   Deviance:                       3373.7
Time:                             13:32:56   Pearson chi2:                 4.28e+03
No. Iterations:                         11   Pseudo R-squ. (CS):             0.3161
Covariance Type:                 nonrobust                                         
                                                                 coef    std err          z      P>|z|     